<a href="https://colab.research.google.com/github/meri-crush/Experiment_Learning_AI/blob/main/Basic_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install torch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
torch.manual_seed(1337)

In [ ]:
text = "hello world"

In [ ]:
chars = sorted(list(set(text)))

In [ ]:
vocab_size = len(chars)

In [ ]:
# Numbers of tokens in the language
vocab_size

8

In [ ]:
# We convert characters into numbers and this is called tokenization

In [ ]:
stoi = { ch:i for i , ch in enumerate(chars)}

In [ ]:
print(stoi)

{' ': 0, 'd': 1, 'e': 2, 'h': 3, 'l': 4, 'o': 5, 'r': 6, 'w': 7}


In [ ]:
itos = {i:ch for i, ch in enumerate(chars)}

In [ ]:
itos

{0: ' ', 1: 'd', 2: 'e', 3: 'h', 4: 'l', 5: 'o', 6: 'r', 7: 'w'}

In [ ]:
def encode(s):
  return [stoi[c] for c in s]

In [ ]:
encode("hello")

[3, 2, 4, 4, 5]

In [ ]:
def decode(l):
  return ''.join([itos[i] for i in l])

In [ ]:
decode([3,2,4,4,5])

'hello'

In [ ]:
encode("hello")

[3, 2, 4, 4, 5]

In [ ]:
decode([3,2,4,4,5])

'hello'

In [ ]:
decode(encode("hello"))

'hello'

In [ ]:
data = torch.tensor(encode(text), dtype = torch.long)

In [ ]:
encode("hello world")

[3, 2, 4, 4, 5, 0, 7, 5, 6, 4, 1]

In [ ]:
torch.tensor([3, 2, 4, 4, 5, 0, 7, 5, 6, 4, 1])

tensor([3, 2, 4, 4, 5, 0, 7, 5, 6, 4, 1])

In [ ]:
print(data)

tensor([3, 2, 4, 4, 5, 0, 7, 5, 6, 4, 1])


In [ ]:
len(data)

11

In [ ]:
block_size = 4

In [ ]:
z = data[:block_size]

In [ ]:
y = data[1:block_size+1]

In [ ]:
print(y,z)

tensor([2, 4, 4, 5]) tensor([3, 2, 4, 4])


In [ ]:
n_embd = 8

In [ ]:
token_embedding_table = nn.Embedding(vocab_size, n_embd)

In [ ]:
token_embedding_table

Embedding(8, 8)

In [ ]:
x = [3,2,4,4]

In [ ]:
emb = token_embedding_table(torch.tensor(x))

In [ ]:
emb.shape

torch.Size([4, 8])

In [ ]:
lm_head = nn.Linear(n_embd, vocab_size)

In [ ]:
lm_head

Linear(in_features=8, out_features=8, bias=True)

In [ ]:
logits = lm_head(emb)

In [ ]:
logits

tensor([[ 0.5352,  0.1111, -0.4425,  0.6983,  0.1974,  1.0948,  1.1986,  1.7776],
        [-0.4068, -0.6961,  0.0480, -0.0645,  1.1288,  0.6118,  1.0966, -0.3792],
        [ 0.0823, -0.0980,  0.0874, -0.4881,  0.1211, -0.3325, -0.9135, -0.3252],
        [ 0.0823, -0.0980,  0.0874, -0.4881,  0.1211, -0.3325, -0.9135, -0.3252]],
       grad_fn=<AddmmBackward0>)

In [ ]:
logits = lm_head(emb)

In [ ]:
logits.shape

torch.Size([4, 8])

In [ ]:
loss = F.cross_entropy(logits, torch.tensor(y))

/tmp/ipykernel_153/1409619206.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  loss = F.cross_entropy(logits, torch.tensor(y))


In [ ]:
loss

tensor(2.1811, grad_fn=<NllLossBackward0>)

In [ ]:
loss.backward()

In [ ]:
optimizer = torch.optim.AdamW(list(token_embedding_table.parameters()) + list(lm_head.parameters()), lr=1e-3)

In [ ]:
optimizer.step()

In [ ]:
optimizer.zero_grad()

In [ ]:
for step in range(1000):
    optimizer.zero_grad()
    emb = token_embedding_table(torch.tensor(x))
    logits = lm_head(emb)
    loss = F.cross_entropy(logits, torch.tensor(y))
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(step, loss.item())

/tmp/ipykernel_153/726702844.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  loss = F.cross_entropy(logits, torch.tensor(y))


0 2.1719424724578857
100 1.389385461807251
200 0.8551285862922668
300 0.5869221687316895
400 0.4750577211380005
500 0.42464348673820496
600 0.3986681401729584
700 0.38371706008911133
800 0.3743618130683899
900 0.36812663078308105


In [ ]:
idx = torch.tensor([encode("h")])

In [ ]:
emb = token_embedding_table(idx)

In [ ]:
emb

tensor([[[ 0.5534, -2.1367, -0.8693, -0.8679,  0.9978, -0.4807,  2.1856,
           2.4010]]], grad_fn=<EmbeddingBackward0>)

In [ ]:
logits = lm_head(emb)

In [ ]:
logits = logits[:, -1, :]

In [ ]:
probs = F.softmax(logits, dim=-1)

In [ ]:
idx_next = torch.multinomial(probs, num_samples = 1)

In [ ]:
idx_next

tensor([[2]])

In [ ]:
decode([idx_next.item()])

'e'

In [ ]:
decode([idx_next.item()])

'e'

In [ ]:
idx = torch.cat((idx, idx_next), dim=1)

In [ ]:
idx

tensor([[3, 2]])

In [ ]:
decode(idx[0].tolist())

'he'

In [ ]:
for _ in range(10):
    emb = token_embedding_table(idx)
    logits = lm_head(emb)
    logits = logits[:, -1, :]
    probs = F.softmax(logits, dim=-1)
    idx_next = torch.multinomial(probs, num_samples=1)
    idx = torch.cat((idx, idx_next), dim=1)

In [ ]:
logits = logits[-1]

In [ ]:
print(decode(idx[0].tolist()))

hello elolol
